<a href="https://colab.research.google.com/github/sadhvik02/Adaptive_Hierarchical_Cyber_Attack_Detection/blob/main/Semantic-Search-on-Twitter-API-Documentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install sentence-transformers chromadb


In [5]:
%cd /content
!git clone https://github.com/xdevplatform/postman-twitter-api.git twitter_docs
!ls twitter_docs


/content
Cloning into 'twitter_docs'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 65 (delta 9), reused 0 (delta 0), pack-reused 53 (from 1)
Receiving objects: 100% (65/65), 125.58 KiB | 4.65 MiB/s, done.
Resolving deltas: 100% (31/31), done.
 CODE_OF_CONDUCT.md   LICENSE	 'Twitter API v2.postman_collection.json'
 CONTRIBUTING.md      README.md  'Twitter API v2.postman_environment.json'


In [6]:
%%writefile index_builder.py
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # force CPU

import os as _os
from typing import List, Tuple

import chromadb
from sentence_transformers import SentenceTransformer

DOCS_DIR = "twitter_docs"   # GitHub repo we just cloned
DB_DIR = "chroma_db"
COLLECTION_NAME = "twitter_api_docs"

CHUNK_SIZE = 800   # characters
CHUNK_OVERLAP = 200


def iter_documents(root: str):
    """
    Walk the docs directory and yield (path, text) for useful files.
    We index .json, .md, .txt (includes the Postman collection JSON).
    """
    for dirpath, _, filenames in _os.walk(root):
        for fname in filenames:
            lower = fname.lower()
            if lower.endswith((".json", ".md", ".txt")):
                path = _os.path.join(dirpath, fname)
                try:
                    with open(path, "r", encoding="utf-8", errors="ignore") as f:
                        text = f.read()
                    if text.strip():
                        yield path, text
                except Exception:
                    continue


def chunk_text(text: str) -> List[str]:
    """
    Simple character-based chunking with overlap.
    """
    text = text.replace("\r\n", "\n")
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + CHUNK_SIZE, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks


def build_index():
    # Ensure DB folder exists
    _os.makedirs(DB_DIR, exist_ok=True)

    # Load embedding model
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Connect to ChromaDB
    client = chromadb.PersistentClient(path=DB_DIR)

    # Clean old collection if exists
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.get_or_create_collection(COLLECTION_NAME)

    all_chunks = []
    all_ids = []
    all_metadatas = []

    doc_id = 0
    for path, text in iter_documents(DOCS_DIR):
        chunks = chunk_text(text)
        for i, ch in enumerate(chunks):
            chunk_id = f"{doc_id}_{i}"
            all_ids.append(chunk_id)
            all_chunks.append(ch)
            all_metadatas.append({"source": path})
        doc_id += 1

    print(f"Total chunks: {len(all_chunks)}")

    if not all_chunks:
        print("No chunks found, index will be empty.")
        return

    # Embed in batches
    BATCH_SIZE = 64
    for start in range(0, len(all_chunks), BATCH_SIZE):
        batch_chunks = all_chunks[start:start + BATCH_SIZE]
        batch_ids = all_ids[start:start + BATCH_SIZE]
        batch_metas = all_metadatas[start:start + BATCH_SIZE]

        embeddings = model.encode(batch_chunks)
        collection.add(
            ids=batch_ids,
            embeddings=embeddings,
            documents=batch_chunks,
            metadatas=batch_metas,
        )

    print("Index built successfully!")


if __name__ == "__main__":
    build_index()


Writing index_builder.py


In [7]:
%%writefile semantic_search.py
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # force CPU

import argparse
import json

import chromadb
from sentence_transformers import SentenceTransformer

DB_DIR = "chroma_db"
COLLECTION_NAME = "twitter_api_docs"


def search(query: str, top_k: int = 5):
    # Load model
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Connect to Chroma
    client = chromadb.PersistentClient(path=DB_DIR)
    collection = client.get_collection(COLLECTION_NAME)

    # Embed query
    q_emb = model.encode([query])

    # Query vector store
    results = collection.query(
        query_embeddings=q_emb,
        n_results=top_k,
        include=["documents", "distances", "metadatas"],
    )

    docs = results.get("documents", [[]])[0]
    dists = results.get("distances", [[]])[0]
    metas = results.get("metadatas", [[]])[0]

    output = {
        "query": query,
        "results": [
            {
                "rank": i + 1,
                "score": float(dists[i]),
                "text": docs[i],
                "source": metas[i].get("source", "") if isinstance(metas[i], dict) else ""
            }
            for i in range(len(docs))
        ],
    }

    print(json.dumps(output, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--query", required=True, help="Search query")
    parser.add_argument("--top_k", type=int, default=5, help="Number of results to return")
    args = parser.parse_args()

    search(args.query, top_k=args.top_k)


Writing semantic_search.py


In [9]:
!python semantic_search.py --query "how to authenticate"


2025-11-17 09:11:22.799263: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763370682.875993    2940 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763370682.901232    2940 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763370682.970351    2940 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763370682.970423    2940 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763370682.970429    2940 computation_placer.cc:177] computation placer alr